# 05 — LLM et RAG (sans clé API)

**Objectif :** comprendre les embeddings et construire un moteur de recherche sémantique — la base de tout RAG.
Tout fonctionne **hors ligne** (le modèle d'embeddings se télécharge une fois, ~120 Mo).

In [ ]:
# pip install sentence-transformers
from sentence_transformers import SentenceTransformer, util

modele = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
phrases = [
    "Le sorgho se plante après les premières pluies.",
    "On sème le sorgho au début de la saison des pluies.",
    "Le prix du carburant a augmenté.",
    "Appliquer de la fumure organique améliore les rendements.",
]
emb = modele.encode(phrases, normalize_embeddings=True)
print("Forme des embeddings :", emb.shape)   # (4, 384)


In [ ]:
# Similarités 2 à 2 : les phrases 0 et 1 doivent être proches, la 2 éloignée
sim = util.cos_sim(emb, emb)
for i in range(len(phrases)):
    for j in range(i + 1, len(phrases)):
        print(f"{sim[i][j]:.3f} | {phrases[i][:35]} ↔ {phrases[j][:35]}")


In [ ]:
# Recherche sémantique : la base d'un RAG
documents = [
    "La fumure organique améliore le rendement du mil de 20 à 40 % au Sahel.",
    "Le niébé fixe l'azote et enrichit le sol pour la culture suivante.",
    "Le stockage hermétique protège les grains contre les charançons.",
    "Le zaï récupère les sols dégradés en captant l'eau de ruissellement.",
    "Le sorgho résiste mieux à la sécheresse que le maïs.",
    "La rotation des cultures limite l'épuisement des sols.",
]
emb_docs = modele.encode(documents, normalize_embeddings=True)

def chercher(question, k=3):
    q = modele.encode([question], normalize_embeddings=True)
    scores = util.cos_sim(q, emb_docs)[0]
    indices = scores.argsort(descending=True)[:k]
    return [(documents[i], float(scores[i])) for i in indices]

# TODO : teste 3 questions de ton choix sur l'agriculture sahélienne
for question in ["Comment améliorer mes rendements ?",
                 "Comment protéger mes grains stockés ?",
                 "Quelle culture résiste à la sécheresse ?"]:
    print("\nQ :", question)
    for doc, s in chercher(question):
        print(f"   [{s:.3f}] {doc}")


In [ ]:
# Construction du prompt RAG (à envoyer ensuite à un LLM : API ou Ollama en local)
def construire_prompt(question, k=3):
    extraits = chercher(question, k)
    contexte = "\n".join(f"[{i+1}] {d}" for i, (d, _) in enumerate(extraits))
    return f"""Tu réponds en français, uniquement à partir du contexte.
Si l'information n'y est pas, réponds exactement : "Je ne trouve pas cette information."
Cite les sources entre crochets, par exemple [1].

<CONTEXTE>
{contexte}
</CONTEXTE>

Question : {question}
Réponse :""", extraits

prompt, extraits = construire_prompt("Comment protéger mes grains ?")
print(prompt)


## 🛠️ Exercices
1. Ajoute **20 documents** de ton domaine (santé, droit du travail, appels d'offres ONG) et teste 10 questions.
2. Mesure le **rappel de la recherche** : sur 10 questions dont tu connais la bonne réponse, combien de fois le bon document est-il dans le top 3 ? (C'est la métrique clé d'un RAG.)
3. Compare 2 découpages : phrases entières vs morceaux de 200 caractères. Lequel retrouve le mieux les bonnes réponses ?
4. **Si tu as une clé API** (ou Ollama en local) : envoie le prompt et vérifie que le modèle **refuse** de répondre à une question hors contexte (test anti-hallucination).

### ✅ Auto-évaluation
- [ ] Je sais expliquer ce qu'est un embedding
- [ ] Je sais construire une recherche sémantique
- [ ] Je sais écrire un prompt RAG avec citations et droit de refus
- [ ] Je sais mesurer la qualité de ma recherche séparément de la génération
